# CS781 Capstone Phase 7: Prior Case Retrieval (PCR)
## Submission 08: Master Ensemble via Reciprocal Rank Fusion (RRF) & Lexical BM25 Stacking [FINAL SUBMISSION]

* **Task**: Prior Case Retrieval (PCR) — Retrieve Top 7 legally & semantically relevant prior cases.
* **Target Metric**: **Micro-F1@7** (Secondary: Micro-Precision@7, Micro-Recall@7).
* **Ensemble Strategy**: **Reciprocal Rank Fusion (RRF)** ([Cormack, Clarke, and Büttcher, SIGIR 2009](https://dl.acm.org/doi/10.1145/1571941.1572114)):
  $$RRF(d \in D_q) = \sum_{m \in \mathcal{M}} w_m \cdot \frac{1}{k + r_m(d)}$$
  where $k = 60$, $r_m(d)$ is the 1-based rank of candidate case $d$ under model $m$, and $w_m$ is the model's reliability weight.
* **Ensemble Model Roster (Diverse Model Family)**:
  1. **Sub-01 (`weight = 1.25`)**: Model 4 Full FT Hierarchical Bi-Encoder ($154\text{M}$, 14 layers, $d=896$, InfoNCE)
  2. **Sub-02 (`weight = 1.15`)**: Model 1 True Multiplicative LoRMA ($135\text{M}$, $r=16$, Triplet Margin + BM25 Hard Negs)
  3. **Sub-03 (`weight = 1.15`)**: Model 2 Wide Mistral QLoRA 4-bit ($136\text{M}$, NF4 + Attention-Weighted MLP Pooling)
  4. **Sub-04 (`weight = 1.00`)**: Model 1 AdaLoRA Dynamic SVD Pruning ($135\text{M}$, MarginMSE Soft Overlap)
  5. **Sub-05 (`weight = 1.05`)**: Model 2 Wide Mistral Houlsby Adapters ($136\text{M}$, Series Bottleneck + CoSENT $\lambda=20$)
  6. **Sub-06 (`weight = 1.00`)**: Model 3 Compact Distilled Student ($110\text{M}$, Full MHA, Knowledge Distillation)
  7. **Sub-07 (`weight = 1.25`)**: Model 4 Poly-Encoder Multi-Vector ($154\text{M}$, $M=8$ Aspect Codebook Vectors)
  8. **BM25 Lexical (`weight = 1.00`)**: Standalone BM25Okapi ($k_1=1.5, b=0.75$) with legal stopword filtering for exact statutory citations & case names.
* **Fault-Tolerant Resilience**: Automatically pulls available `prediction.json` files from `sarweshbarnwal/CS781-Legal-Models` under `Phase_7/`. If any individual model has not completed yet, it gracefully ensembles all currently available models plus the local BM25 retriever without crashing.
* **Strict CodaBench Output**: Validates exactly 7 unique candidate IDs per query from `test_candidates.json`, packages root `submission.zip`, plots model diversity heatmaps, and syncs artifacts to Hugging Face.


In [ ]:
# 1. Environment Dependencies & Hugging Face Authentication
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys
import gc
import json
import time
import math
import glob
import random
import zipfile
import shutil
import re
from collections import defaultdict, Counter
from typing import List, Dict, Tuple, Optional, Any, Set

import numpy as np

# Kaggle / Colab / Local HF Hub Authentication
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    print("✓ Loaded HF_TOKEN from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN", None)
    if HF_TOKEN:
        print("✓ Loaded HF_TOKEN from environment variable.")
    else:
        print("⚠ HF_TOKEN not found in secrets or env; please specify before writing to private repos.")

from huggingface_hub import HfApi, login, snapshot_download, hf_hub_download
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=True)
    print("✓ Hugging Face authentication successful.")
else:
    print("⚠ Running without Hugging Face login; authenticated pushes will be disabled.")


In [ ]:
# 2. Execution Profile Selection & Global Configuration
PROFILE = "CORE"  # Options: "SMOKE", "CORE"

PROFILES = {
    "SMOKE": {
        "max_test_queries": 20,
        "max_candidates": 100,
        "rrf_k": 60,
        "description": "Fast Functional Verification (Smoke Mode)"
    },
    "CORE": {
        "max_test_queries": None,     # Full test set
        "max_candidates": None,        # Full candidate pool
        "rrf_k": 60,                   # Standard RRF constant (Cormack et al., 2009)
        "description": "Full Production Master Ensembling (Core Mode)"
    }
}

CFG = PROFILES[PROFILE]
print(f"=== Active Profile: {PROFILE} ({CFG['description']}) ===")

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MASTER_REPO = "sarweshbarnwal/CS781-Legal-Models"
EXPERIMENT_NAME = "sub08_master_ensemble_rrf"
HUB_SUBFOLDER = f"Phase_7/{EXPERIMENT_NAME}"

OUTPUT_DIR = "/kaggle/working/phase7_sub08" if os.path.exists("/kaggle") else "./phase7_sub08"
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
HEATMAP_PLOT_PATH = os.path.join(OUTPUT_DIR, "ensemble_agreement_heatmap.png")
REPORT_JSON_PATH = os.path.join(OUTPUT_DIR, "ensemble_report.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Local artifacts directory: {OUTPUT_DIR}")
print(f"HF Hub destination: {HF_MASTER_REPO}/{HUB_SUBFOLDER}")


In [ ]:
# 3. Dataset Ingestion (Test Set & Candidate Pool)
raw_data_dir = os.path.join(OUTPUT_DIR, "raw_data")
os.makedirs(raw_data_dir, exist_ok=True)

print("Downloading Phase 7 PCR dataset from Hugging Face...")
snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    local_dir=raw_data_dir,
    allow_patterns=["Phase_7_PCR/Test/*", "Phase_7_PCR/test*", "Phase_7_PCR/*.json"],
    token=HF_TOKEN
)

pcr_base = os.path.join(raw_data_dir, "Phase_7_PCR")
if not os.path.exists(pcr_base):
    for root, dirs, files in os.walk(raw_data_dir):
        if "test.json" in files:
            pcr_base = root
            break

print(f"PCR Data Directory: {pcr_base}")

def load_json_file(filename: str) -> dict:
    candidates = [
        os.path.join(pcr_base, filename),
        os.path.join(pcr_base, "Test", filename),
        os.path.join(pcr_base, "Train", filename),
    ]
    for path in candidates:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
    for root, _, files in os.walk(pcr_base):
        if filename in files:
            with open(os.path.join(root, filename), "r", encoding="utf-8") as f:
                return json.load(f)
    raise FileNotFoundError(f"Could not locate {filename} under {pcr_base}")

test_queries_raw = load_json_file("test.json")
test_candidates_raw = load_json_file("test_candidates.json")

print(f"Loaded Raw Test Queries:    {len(test_queries_raw):,}")
print(f"Loaded Raw Test Candidates: {len(test_candidates_raw):,}")

def flatten_case_text(case_obj: Any) -> str:
    if isinstance(case_obj, str):
        return case_obj.strip()
    elif isinstance(case_obj, list):
        return " ".join([str(x).strip() for x in case_obj if str(x).strip()])
    elif isinstance(case_obj, dict):
        if "text" in case_obj:
            return flatten_case_text(case_obj["text"])
        vals = [flatten_case_text(v) for v in case_obj.values() if isinstance(v, (str, list))]
        return " ".join(vals)
    return str(case_obj).strip()

def process_query_dict(q_dict: dict, max_count: Optional[int] = None) -> List[dict]:
    items = []
    for qid, qdata in list(q_dict.items())[:max_count]:
        if isinstance(qdata, dict):
            text = flatten_case_text(qdata.get("text", qdata.get("query_text", "")))
        else:
            text = flatten_case_text(qdata)
        items.append({"qid": str(qid), "text": text})
    return items

def process_candidate_dict(c_dict: dict, max_count: Optional[int] = None) -> Dict[str, str]:
    limit = max(7, max_count) if max_count is not None else None
    processed = {}
    for cid, cdata in list(c_dict.items())[:limit]:
        processed[str(cid)] = flatten_case_text(cdata)
    return processed

test_queries = process_query_dict(test_queries_raw, CFG["max_test_queries"])
test_candidates = process_candidate_dict(test_candidates_raw, CFG["max_candidates"])

test_qids = [q["qid"] for q in test_queries]
test_cids = list(test_candidates.keys())

print(f"Ensemble Execution Counts: {len(test_queries):,} queries, {len(test_candidates):,} candidate pool cases.")


In [ ]:
# 4. Standalone High-Performance Lexical BM25 Engine
import math
from collections import Counter

LEGAL_STOPWORDS = {
    "the", "of", "and", "in", "to", "a", "is", "that", "for", "it", "as", "was", "with",
    "be", "by", "on", "not", "he", "at", "this", "from", "court", "case", "appellant",
    "respondent", "held", "learned", "counsel", "order", "act", "section", "dated",
    "judgment", "petition", "appeal", "high", "supreme", "bench", "state", "union",
    "law", "article", "rule", "provisions", "clause", "sub", "para", "v.", "versus",
    "an", "or", "which", "are", "were", "been", "have", "has", "had", "would", "could"
}

def tokenize_legal_text(text: str) -> List[str]:
    # Extract legal citations, statutory sections (e.g. s.302, art.21, 138-ni), and keywords
    raw_tokens = re.findall(r'[a-zA-Z0-9_\-\.]+', text.lower())
    return [t for t in raw_tokens if len(t) > 2 and t not in LEGAL_STOPWORDS]

class BM25Retriever:
    """
    BM25Okapi retriever optimized for legal case law matching:
    - Sublinear term frequency scaling
    - Exact citation & statutory provision preservation
    - Pre-computed IDF across candidate corpus
    """
    def __init__(self, candidates: Dict[str, str], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.doc_ids = list(candidates.keys())
        self.doc_count = len(self.doc_ids)
        self.doc_lengths = {}
        self.doc_term_freqs = {}
        self.doc_freqs = Counter()

        print(f"Indexing {self.doc_count:,} candidate cases with BM25...")
        total_len = 0
        for doc_id, text in candidates.items():
            tokens = tokenize_legal_text(text)
            self.doc_lengths[doc_id] = len(tokens)
            total_len += len(tokens)
            tf = Counter(tokens)
            self.doc_term_freqs[doc_id] = tf
            for t in tf.keys():
                self.doc_freqs[t] += 1

        self.avg_dl = max(total_len / max(self.doc_count, 1), 1.0)
        print(f"✓ BM25 Index Built: Vocab Size: {len(self.doc_freqs):,} | Avg Doc Length: {self.avg_dl:.1f} tokens")

        # Precompute IDF
        self.idf = {}
        for term, df in self.doc_freqs.items():
            self.idf[term] = math.log(1.0 + (self.doc_count - df + 0.5) / (df + 0.5))

    def retrieve(self, query_text: str, top_k: int = 50) -> List[Tuple[str, float]]:
        q_tokens = tokenize_legal_text(query_text)
        if not q_tokens:
            return [(cid, 0.0) for cid in self.doc_ids[:top_k]]

        q_tf = Counter(q_tokens)
        scores = defaultdict(float)

        for term, q_count in q_tf.items():
            if term not in self.idf:
                continue
            idf_val = self.idf[term]
            for doc_id in self.doc_ids:
                tf_val = self.doc_term_freqs[doc_id].get(term, 0)
                if tf_val > 0:
                    dl = self.doc_lengths[doc_id]
                    denom = tf_val + self.k1 * (1.0 - self.b + self.b * (dl / self.avg_dl))
                    term_score = idf_val * (tf_val * (self.k1 + 1.0)) / max(denom, 1e-9)
                    scores[doc_id] += term_score

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        # Fallback if fewer candidates scored
        if len(ranked) < top_k:
            ranked_set = {cid for cid, _ in ranked}
            for cid in self.doc_ids:
                if cid not in ranked_set:
                    ranked.append((cid, 0.0))
                if len(ranked) >= top_k:
                    break
        return ranked[:top_k]

print("Initializing candidate corpus BM25 retriever...")
bm25_engine = BM25Retriever(test_candidates)

# Pre-compute BM25 predictions for all test queries
print(f"Computing BM25 rankings for {len(test_queries):,} test queries...")
bm25_predictions = {}
for q in test_queries:
    qid = q["qid"]
    ranked_pairs = bm25_engine.retrieve(q["text"], top_k=min(50, len(test_candidates)))
    bm25_predictions[qid] = [cid for cid, _ in ranked_pairs]

print(f"✓ BM25 retrieval completed for {len(bm25_predictions):,} queries.")


In [ ]:
# 5. Remote Neural Model Predictions Fetcher from Hugging Face Master Hub
from huggingface_hub import list_repo_files

MODEL_CONFIGS = [
    {
        "id": "sub01",
        "name": "Sub-01: Model 4 Full FT Hierarchical Bi-Encoder (154M)",
        "subfolder": "Phase_7/sub01_model4_full_ft_hierarchical",
        "weight": 1.25,
    },
    {
        "id": "sub02",
        "name": "Sub-02: Model 1 True Multiplicative LoRMA (135M)",
        "subfolder": "Phase_7/sub02_model1_true_lorma_hardneg",
        "weight": 1.15,
    },
    {
        "id": "sub03",
        "name": "Sub-03: Model 2 Wide Mistral QLoRA 4-bit (136M)",
        "subfolder": "Phase_7/sub03_model2_qlora_attention_pool",
        "weight": 1.15,
    },
    {
        "id": "sub04",
        "name": "Sub-04: Model 1 AdaLoRA Dynamic SVD (135M)",
        "subfolder": "Phase_7/sub04_model1_adalora_margin_mse",
        "weight": 1.00,
    },
    {
        "id": "sub05",
        "name": "Sub-05: Model 2 Houlsby Series Adapters (136M)",
        "subfolder": "Phase_7/sub05_model2_houlsby_cosent",
        "weight": 1.05,
    },
    {
        "id": "sub06",
        "name": "Sub-06: Model 3 Compact Distilled Student (110M)",
        "subfolder": "Phase_7/sub06_model4_to_model3_distill",
        "weight": 1.00,
    },
    {
        "id": "sub07",
        "name": "Sub-07: Model 4 Poly-Encoder Multi-Vector (154M)",
        "subfolder": "Phase_7/sub07_model4_poly_encoder_multiaspect",
        "weight": 1.25,
    },
    {
        "id": "bm25",
        "name": "BM25: Lexical Citation & Statute Retriever",
        "subfolder": None,  # Computed directly in-memory
        "weight": 1.00,
    }
]

model_predictions = {}
# Add BM25 predictions first
model_predictions["bm25"] = bm25_predictions

print("=== Fetching Available Neural Predictions from Hugging Face Hub ===")
dl_preds_dir = os.path.join(OUTPUT_DIR, "downloaded_predictions")
os.makedirs(dl_preds_dir, exist_ok=True)

for m in MODEL_CONFIGS:
    mid = m["id"]
    if mid == "bm25":
        continue

    subfolder = m["subfolder"]
    print(f"Checking for {m['name']} at {HF_MASTER_REPO}/{subfolder}...")
    local_m_dir = os.path.join(dl_preds_dir, mid)
    os.makedirs(local_m_dir, exist_ok=True)

    try:
        fpath = hf_hub_download(
            repo_id=HF_MASTER_REPO,
            filename=f"{subfolder}/prediction.json",
            local_dir=local_m_dir,
            token=HF_TOKEN
        )
        with open(fpath, "r", encoding="utf-8") as pf:
            preds = json.load(pf)
            model_predictions[mid] = preds
            print(f"  ✓ Loaded predictions for {mid}: {len(preds):,} queries found.")
    except Exception as e:
        print(f"  ℹ Notice: {mid} prediction not yet found on Hub ({e}). Will proceed gracefully without it.")

active_models = list(model_predictions.keys())
print(f"\n=== Active Models in Master Ensemble: {len(active_models)} ===")
for mid in active_models:
    w = next(m["weight"] for m in MODEL_CONFIGS if m["id"] == mid)
    mname = next(m["name"] for m in MODEL_CONFIGS if m["id"] == mid)
    print(f"  * [{mid.upper()}] (Weight: {w:.2f}) -> {mname}")


In [ ]:
# 6. Reciprocal Rank Fusion (RRF, k=60) Implementation
def reciprocal_rank_fusion(
    predictions_by_model: Dict[str, Dict[str, List[str]]],
    weights: Dict[str, float],
    query_ids: List[str],
    candidate_pool: List[str],
    k_rrf: int = 60,
    top_n: int = 7
) -> Dict[str, List[str]]:
    """
    Computes Reciprocal Rank Fusion across all available models:
        RRF_score(d in D_q) = sum_{m in Models} w_m * [1 / (k + r_m(d))]
    """
    fused_predictions = {}
    cand_pool_set = set(candidate_pool)

    for qid in query_ids:
        candidate_scores = defaultdict(float)

        for model_id, preds_dict in predictions_by_model.items():
            if qid not in preds_dict:
                continue

            ranked_cids = preds_dict[qid]
            model_weight = weights.get(model_id, 1.0)

            for rank_idx, cid in enumerate(ranked_cids):
                if cid not in cand_pool_set:
                    continue
                rank = rank_idx + 1  # 1-based rank
                score = model_weight * (1.0 / (k_rrf + rank))
                candidate_scores[cid] += score

        # Sort candidate IDs by total RRF score in descending order
        sorted_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
        top_cids = [cid for cid, _ in sorted_candidates[:top_n]]

        # Fallback guarantee: if candidate pool has more items and top_cids < 7
        if len(top_cids) < top_n and candidate_pool:
            for fallback_cid in candidate_pool:
                if fallback_cid not in top_cids:
                    top_cids.append(fallback_cid)
                if len(top_cids) == top_n:
                    break

        fused_predictions[qid] = top_cids[:top_n]

    return fused_predictions

weights_dict = {m["id"]: m["weight"] for m in MODEL_CONFIGS}
print(f"Initialized RRF Engine with k={CFG['rrf_k']} and weights: {weights_dict}")


In [ ]:
# 7. Pre-flight Verification & Ranking Assertion
print("=== Running Pre-flight Fusion Logic Assertion ===")
sample_qids = test_qids[:min(5, len(test_qids))]

sample_fused = reciprocal_rank_fusion(
    predictions_by_model=model_predictions,
    weights=weights_dict,
    query_ids=sample_qids,
    candidate_pool=test_cids,
    k_rrf=CFG["rrf_k"],
    top_n=7
)

assert len(sample_fused) == len(sample_qids), f"ERROR: Sample query count mismatch: {len(sample_fused)} vs {len(sample_qids)}"
test_cids_set = set(test_cids)

for sqid, scids in sample_fused.items():
    assert len(scids) == 7, f"ERROR: Sample query {sqid} did not yield exactly 7 candidates (got {len(scids)})"
    assert len(set(scids)) == 7, f"ERROR: Sample query {sqid} yielded duplicate candidates: {scids}"
    for sc in scids:
        assert sc in test_cids_set, f"ERROR: Candidate {sc} is not in candidate pool!"

print(f"✓ Pre-flight Assertion PASSED on {len(sample_qids)} queries.")
print(f"  Sample Prediction for Query [{sample_qids[0]}]:")
print(f"    Top 7 RRF Candidates: {sample_fused[sample_qids[0]]}")


In [ ]:
# 8. Execute Full Master Ensembling across All Test Queries
print(f"Executing Reciprocal Rank Fusion on {len(test_qids):,} queries using {len(active_models)} models...")
start_time = time.time()

master_predictions = reciprocal_rank_fusion(
    predictions_by_model=model_predictions,
    weights=weights_dict,
    query_ids=test_qids,
    candidate_pool=test_cids,
    k_rrf=CFG["rrf_k"],
    top_n=7
)

elapsed = time.time() - start_time
print(f"✓ Master Ensembling complete in {elapsed:.2f} seconds ({len(test_qids) / max(elapsed, 1e-4):.1f} queries/sec).")


In [ ]:
# 9. Model Diversity & Agreement Analysis (Pairwise Jaccard Similarity)
import matplotlib.pyplot as plt

active_keys = [m for m in model_predictions.keys() if len(model_predictions[m]) > 0]
n_active = len(active_keys)

if n_active >= 2:
    print(f"Computing pairwise Top-7 Jaccard overlap matrix across {n_active} models...")
    jaccard_matrix = np.zeros((n_active, n_active))

    for i in range(n_active):
        m1 = active_keys[i]
        for j in range(n_active):
            m2 = active_keys[j]
            jaccards = []
            for qid in test_qids:
                if qid in model_predictions[m1] and qid in model_predictions[m2]:
                    s1 = set(model_predictions[m1][qid][:7])
                    s2 = set(model_predictions[m2][qid][:7])
                    intersection = len(s1.intersection(s2))
                    union = len(s1.union(s2))
                    jaccards.append(intersection / max(union, 1))
            jaccard_matrix[i, j] = np.mean(jaccards) if jaccards else 0.0

    plt.figure(figsize=(9, 7))
    plt.imshow(jaccard_matrix, cmap="Purples", vmin=0.0, vmax=1.0)
    plt.colorbar(label="Average Top-7 Jaccard Overlap")
    plt.title("Phase 7 Model Diversity & Retrieval Agreement Heatmap", fontsize=12, fontweight="bold")

    tick_labels = [k.upper() for k in active_keys]
    plt.xticks(range(n_active), tick_labels, rotation=45, ha="right")
    plt.yticks(range(n_active), tick_labels)

    for i in range(n_active):
        for j in range(n_active):
            plt.text(j, i, f"{jaccard_matrix[i, j]:.2f}", ha="center", va="center",
                     color="white" if jaccard_matrix[i, j] > 0.5 else "black", fontsize=10)

    plt.tight_layout()
    plt.savefig(HEATMAP_PLOT_PATH, dpi=300)
    plt.show()
    print(f"✓ Saved agreement heatmap to: {HEATMAP_PLOT_PATH}")
else:
    print("Fewer than 2 models available; skipping diversity heatmap.")


In [ ]:
# 10. Strict CodaBench Output Verification
print("=== Strict CodaBench Verification on Master Ensemble ===")

# Save prediction.json
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(master_predictions, f, indent=2)
print(f"✓ Saved prediction.json to: {PREDICTION_PATH}")

# Verification 1: File existence and non-emptiness
assert os.path.exists(PREDICTION_PATH), "ERROR: prediction.json not found!"
with open(PREDICTION_PATH, "r", encoding="utf-8") as f:
    loaded_preds = json.load(f)

# Verification 2: Query count matches official test queries exactly
assert len(loaded_preds) == len(test_queries), f"ERROR: Query count mismatch ({len(loaded_preds)} vs {len(test_queries)})!"

# Verification 3: Exactly 7 unique candidate IDs per query from test_candidates.json
test_cids_set = set(test_candidates.keys())
for qid, cids in loaded_preds.items():
    assert len(cids) == 7, f"ERROR: Query {qid} does not have exactly 7 candidates (got {len(cids)})!"
    assert len(set(cids)) == 7, f"ERROR: Query {qid} contains duplicate candidate IDs!"
    for cid in cids:
        assert cid in test_cids_set, f"ERROR: Candidate {cid} for query {qid} is not in test_candidates.json!"

print("✓ CodaBench Assertions PASSED:")
print(f"  - Total Queries Predicted: {len(loaded_preds):,}")
print("  - Exactly 7 unique candidates per query: VERIFIED.")
print("  - Zero duplicate IDs: VERIFIED.")
print("  - All candidates belong to official test_candidates.json: VERIFIED.")


In [ ]:
# 11. Packaging submission.zip & Root Level Verification
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print(f"✓ Successfully packaged submission.zip: {SUBMISSION_ZIP}")

# Verify archive structure: prediction.json MUST sit directly at root
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as check_zip:
    namelist = check_zip.namelist()
    assert namelist == ["prediction.json"], f"ERROR: submission.zip root structure invalid: {namelist}"

print("✓ Root packaging verified: 'prediction.json' is at the root level without parent folders.")


In [ ]:
# 12. Master Hugging Face Hub Synchronization
# Build ensemble summary metadata
report_data = {
    "experiment": EXPERIMENT_NAME,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "profile": PROFILE,
    "total_queries_predicted": len(master_predictions),
    "rrf_k": CFG["rrf_k"],
    "active_models": active_models,
    "model_weights": {mid: weights_dict[mid] for mid in active_models},
    "codabench_verified": True
}

with open(REPORT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(report_data, f, indent=2)

print("Saved ensemble execution metadata.")

# Push final submission artifacts to Master HF Hub under Phase_7/sub08_master_ensemble_rrf/
if HF_TOKEN:
    try:
        api = HfApi()
        print(f"Uploading final Phase 7 Master Ensemble artifacts to {HF_MASTER_REPO}/{HUB_SUBFOLDER}...")
        api.upload_file(
            path_or_fileobj=PREDICTION_PATH,
            path_in_repo=f"{HUB_SUBFOLDER}/prediction.json",
            repo_id=HF_MASTER_REPO,
            token=HF_TOKEN
        )
        api.upload_file(
            path_or_fileobj=SUBMISSION_ZIP,
            path_in_repo=f"{HUB_SUBFOLDER}/submission.zip",
            repo_id=HF_MASTER_REPO,
            token=HF_TOKEN
        )
        if os.path.exists(HEATMAP_PLOT_PATH):
            api.upload_file(
                path_or_fileobj=HEATMAP_PLOT_PATH,
                path_in_repo=f"{HUB_SUBFOLDER}/ensemble_agreement_heatmap.png",
                repo_id=HF_MASTER_REPO,
                token=HF_TOKEN
            )
        api.upload_file(
            path_or_fileobj=REPORT_JSON_PATH,
            path_in_repo=f"{HUB_SUBFOLDER}/ensemble_report.json",
            repo_id=HF_MASTER_REPO,
            token=HF_TOKEN
        )
        print("✓ All final Phase 7 Master Ensemble artifacts successfully uploaded to Master Repository.")
    except Exception as e:
        print(f"⚠ Master Hub upload failed: {e}")
else:
    print("⚠ HF_TOKEN not present. Skipping final master upload.")

print("\n=======================================================")
print("  CS781 PHASE 7 MASTER ENSEMBLE EXECUTION COMPLETE!   ")
print(f"  Submission File Ready: {SUBMISSION_ZIP}")
print("=======================================================")
